# multivon-eval — Quickstart

[![PyPI](https://img.shields.io/pypi/v/multivon-eval.svg)](https://pypi.org/project/multivon-eval)
[![Docs](https://img.shields.io/badge/docs-evaldocs.multivon.ai-violet)](https://evaldocs.multivon.ai)
[![GitHub](https://img.shields.io/badge/github-multivon--ai%2Fmultivon--eval-black)](https://github.com/multivon-ai/multivon-eval)

**multivon-eval** is an open source Python SDK for evaluating AI outputs — from simple string checks to LLM judges to agent trace analysis.

| Part | What | API key? |
|------|------|----------|
| Setup | API key or local LLM (Ollama, LM Studio) | optional |
| 1 | Plain-English quality checks (`add_check`) | ✓ |
| 2 | Deterministic evaluators — format, length, keywords | ✗ |
| 3 | LLM judge evaluators + built-in model adapters | ✓ |
| 4 | Inspect failures (`failed_cases`, filtering, sampling) | — |
| 5 | Multi-run + flakiness detection | — |
| 6 | Custom adapters — extend for any model client | — |
| 7 | Factory suites — one-line setup for common use cases | — |

**More notebooks:** [RAG Evaluation](rag_eval.ipynb) · [Agent Evaluation](agent_eval.ipynb)

Docs: [evaldocs.multivon.ai](https://evaldocs.multivon.ai) · GitHub: [multivon-ai/multivon-eval](https://github.com/multivon-ai/multivon-eval)

In [ ]:
!pip install 'multivon-eval>=0.7.0' -q

## API key setup

Set one key here. Parts 1 and 3 use it; everything else runs without one.

multivon-eval uses the key for two things: (1) calling your model via the built-in adapters, and (2) the LLM judge that scores outputs. Both default to the smallest/cheapest model available.

In [ ]:
import os

# Set ONE of these
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."   # recommended — uses claude-haiku-4-5
# os.environ["OPENAI_API_KEY"]   = "sk-..."       # uses gpt-4o-mini

### Local LLM (no API key needed for the judge)

Any OpenAI-compatible server works as the judge — Ollama, LM Studio, vLLM, llama.cpp. The model under test and the judge can be different providers.

Set `OPENAI_BASE_URL` in your environment, or pass `base_url=` directly to `JudgeConfig`.

In [ ]:
import os
from multivon_eval import configure, JudgeConfig

# Auto-detect: prefer your cloud key, fall back to a local LLM.
# This way Colab works (cloud) and local Jupyter works (Ollama/LM Studio).
_anthropic = os.environ.get("ANTHROPIC_API_KEY", "")
_openai = os.environ.get("OPENAI_API_KEY", "")

if _anthropic and not _anthropic.startswith("sk-ant-..."):
    # Default Anthropic judge — picked up automatically by the SDK.
    print("Judge: Anthropic claude-haiku-4-5 (from ANTHROPIC_API_KEY)")
elif _openai and not _openai.startswith("sk-..."):
    configure(JudgeConfig(provider="openai", model="gpt-4o-mini"))
    print("Judge: OpenAI gpt-4o-mini (from OPENAI_API_KEY)")
else:
    # Local fallback — Ollama (`ollama pull llama3` first) on default port.
    configure(JudgeConfig(provider="openai", model="llama3",
                          base_url="http://localhost:11434/v1"))
    print("Judge: local Ollama llama3 at :11434 (no cloud key detected)")


---
## Part 1 — Plain-English quality checks

The fastest way to write an eval. Describe what you want in plain English — the SDK auto-generates yes/no questions from your criterion and scores with QAG.

No need to pick an evaluator class or craft questions manually.

In [ ]:
from multivon_eval import EvalSuite, EvalCase

def support_bot(input: str) -> str:
    # Replace with your real model call
    return "You can reset your password by clicking 'Forgot Password' on the login page."

suite = EvalSuite("Support Bot")
suite.add_check("Response explains how to reset the password")
suite.add_check("Tone is professional and not defensive", threshold=0.8)
suite.add_cases([
    EvalCase(input="How do I reset my password?"),
    EvalCase(input="I can't log in, what do I do?"),
])

report = suite.run(support_bot)

In [ ]:
# See what questions the SDK generated from your plain-English criteria.
# `suite.evaluators` is a public read-only view; `ev.criterion` and
# `ev.resolved_questions` are the public accessors on a CheckEvaluator.
for ev in suite.evaluators:
    if hasattr(ev, "criterion"):
        print(f"Criterion: {ev.criterion}")
        for i, q in enumerate(ev.resolved_questions or [], 1):
            print(f"  {i}. {q}")
        print()

In [ ]:
# Pin questions for reproducible CI runs — generated questions vary per run
suite_ci = EvalSuite("Support Bot — CI")
suite_ci.add_check(
    "Response explains how to reset the password",
    questions=[
        "Does the response mention a specific action the user can take?",
        "Does the response mention the 'Forgot Password' link or equivalent?",
        "Does the response avoid introducing unrelated steps?",
    ],
)
suite_ci.add_cases([EvalCase(input="How do I reset my password?")])
report_ci = suite_ci.run(support_bot)

---
## Part 2 — Deterministic evaluators (no API key)

Instant, free, no LLM call. Use for format checks, keyword presence, length constraints, and latency budgets. Mix freely with LLM judges.

In [ ]:
from multivon_eval import (
    EvalSuite, EvalCase,
    NotEmpty, ExactMatch, Contains, WordCount, RegexMatch, BLEU, ROUGE,
)

def qa_model(input: str) -> str:
    answers = {
        "What is the capital of France?": "Paris",
        "What is 2 + 2?":                "4",
        "Name a planet.":                "Mars is a planet in our solar system.",
    }
    return answers.get(input, "I don't know.")

suite = EvalSuite("Deterministic Demo")
suite.add_cases([
    EvalCase(input="What is the capital of France?", expected_output="Paris"),
    EvalCase(input="What is 2 + 2?",                expected_output="4"),
    EvalCase(input="Name a planet.",
             expected_output="Mars is a planet in our solar system."),
])
suite.add_evaluators(
    NotEmpty(),                          # output is not blank
    ExactMatch(),                        # matches expected_output exactly
    Contains(["planet", "Mars"], match_any=True),  # contains at least one keyword
    WordCount(min=1, max=30),            # length in range
    BLEU(),                              # n-gram overlap with expected_output
    ROUGE(),                             # recall-oriented overlap
)

report = suite.run(qa_model)

---
## Part 3 — LLM judge evaluators + model adapters

LLM judge evaluators use a secondary model to assess quality. multivon-eval uses **QAG scoring** — binary yes/no questions instead of unreliable 1–10 ratings. [65% fewer false positives](https://github.com/multivon-ai/multivon-eval/tree/main/benchmarks) than numeric scoring with the same judge model.

### Built-in adapters — no wrapper function needed

`run_with_anthropic()` and `run_with_openai()` call your model directly. No need to define a `my_model` wrapper.

In [ ]:
from multivon_eval import EvalSuite, EvalCase, NotEmpty, Faithfulness, Relevance, Hallucination

suite = EvalSuite("RAG Eval")
suite.add_cases([
    EvalCase(
        input="How do I reset my password?",
        context="Users reset passwords by clicking 'Forgot Password' on the login page.",
    ),
    EvalCase(
        input="What is your refund policy?",
        context="Refunds are processed within 5 business days for eligible orders.",
    ),
])
suite.add_evaluators(
    NotEmpty(),
    Faithfulness(),    # answer stays within the provided context
    Relevance(),       # answer is relevant to the question
    Hallucination(),   # answer doesn't introduce claims not in context
)

# No wrapper function needed — pass the model ID directly
report = suite.run_with_anthropic("claude-haiku-4-5-20251001")
# report = suite.run_with_openai("gpt-4o-mini")

In [ ]:
# Add a system prompt — no wrapper function, no extra boilerplate
report = suite.run_with_anthropic(
    "claude-haiku-4-5-20251001",
    system_prompt="You are a helpful support agent. Answer concisely in one sentence.",
    workers=4,   # all suite.run() params work here
)

---
## Part 4 — Inspect results

The report object is designed for fast iteration on failures.

In [ ]:
# Summary
print(f"Pass rate:   {report.pass_rate:.0%}")
print(f"Avg score:   {report.avg_score:.2f}")
print(f"Total cases: {report.total}")

print("\nScores by evaluator:")
for name, score in report.scores_by_evaluator().items():
    print(f"  {name}: {score:.2f}")

In [ ]:
# See every failure with its reason
print(f"Failed: {len(report.failed_cases)} / {report.total}")

for cr in report.failed_cases:
    print(f"\nInput:  {cr.case_input}")
    print(f"Output: {cr.actual_output[:120]}")
    for r in cr.results:
        if not r.passed:
            print(f"  ✗ {r.evaluator} (score={r.score:.2f})")
            print(f"    {r.reason[:200]}")

In [ ]:
# Filter by a specific evaluator
faithfulness_failures = report.filter_by_evaluator("faithfulness")
print(f"Cases that failed faithfulness: {len(faithfulness_failures)}")

# Random sample — useful for spot-checking large runs without reading every result
sample = report.sample(3, failed_only=True)
print(f"Random sample of {len(sample)} failures for manual review")

In [ ]:
# Save in any format
report.save_json("report.json")   # machine-readable, good for CI diffs
report.save_html("report.html")   # self-contained HTML with per-case breakdown
report.save_csv("report.csv")     # import into Sheets / pandas

print("Saved: report.json  report.html  report.csv")

---
## Part 5 — Multi-run + flakiness detection

LLMs are non-deterministic. A single-run eval score can be noise.
Run each case N times to detect flaky outputs and get statistically meaningful pass rates.

Backed by [NAACL 2025](https://arxiv.org/abs/2502.01775): single-run eval scores are unreliable.

In [ ]:
import random
from multivon_eval import EvalSuite, EvalCase, NotEmpty, ExactMatch

def flaky_model(input: str) -> str:
    # Simulates a non-deterministic model — sometimes right, sometimes blank
    return "Paris" if random.random() > 0.4 else ""

suite = EvalSuite("Flakiness Demo")
suite.add_cases([EvalCase(input="Capital of France?", expected_output="Paris")])
suite.add_evaluators(NotEmpty(), ExactMatch())

report = suite.run(flaky_model, runs=5)  # run each case 5 times

print(f"Flaky cases: {report.flaky_count}")
print(f"Stability:   {report.stability_score:.0%}")
for cr in report.case_results:
    print(f"  pass rate across runs: {cr.run_pass_rate:.0%}  flaky={cr.is_flaky}")

In [ ]:
# early_stop=True uses SPRT to halt each case once the result is statistically clear
# Saves LLM calls on easy cases without sacrificing accuracy
report = suite.run(flaky_model, runs=10, early_stop=True)
print(f"Budget: 10 runs. Avg actually used: ~{report.runs_per_case}")

---
## Part 6 — Custom adapters

Subclass `ModelAdapter` to wrap any model client — add retry logic, prompt templates, structured output parsing, cost tracking, or any other behaviour.

The adapter implements `__call__(input: str) -> str` and plugs directly into `suite.run()`. The built-in adapters (`OpenAIAdapter`, `AnthropicAdapter`) follow the same interface and can also be subclassed.

In [ ]:
from multivon_eval import ModelAdapter

class MyInternalAdapter(ModelAdapter):
    """Wraps an internal model API — swap __call__ for your real client."""

    def __init__(self, endpoint: str, api_key: str):
        self.endpoint = endpoint
        self.api_key = api_key

    def __call__(self, input: str) -> str:
        # Replace with: requests.post(self.endpoint, json={"prompt": input}, ...)
        return f"[stub response for: {input[:40]}]"

report = suite.run(MyInternalAdapter("https://my-model.internal", "my-key"))

In [ ]:
from multivon_eval import AnthropicAdapter

# Extend a built-in adapter to customise prompt construction
class ChainOfThoughtAdapter(AnthropicAdapter):
    """Appends a chain-of-thought instruction to every user message."""

    def _build_messages(self, input: str) -> list[dict]:
        return [{"role": "user", "content": f"{input}\n\nThink step by step."}]

report = suite.run(ChainOfThoughtAdapter("claude-haiku-4-5-20251001"))

In [ ]:
from multivon_eval import OpenAIAdapter

# with_system_prompt() is a quick decorator on any adapter
adapter = OpenAIAdapter("gpt-4o-mini").with_system_prompt(
    "You are a concise support agent. Answer in one sentence."
)
report = suite.run(adapter)

---
## Part 7 — Factory suites

Pre-built suites with the right evaluators for common use cases. Fully composable — add `add_check()` or `add_evaluators()` on top.

In [ ]:
from multivon_eval import EvalSuite, EvalCase

# RAG pipeline — Faithfulness, Hallucination, ContextPrecision, ContextRecall, Relevance
suite = EvalSuite.for_rag()

# Compose: add a plain-English check on top of the pre-built evaluators
suite.add_check("Response cites a specific detail from the source document")

suite.add_cases([
    EvalCase(
        input="What is the refund window?",
        context="Refunds are accepted within 30 days of purchase.",
    ),
])

print("Available factory suites:")
for name in [
    "for_rag()",
    "for_agents()",
    "for_support_bot()",
    "for_summarization()",
    "for_chatbot()",
    "for_classification()",
    "for_document_intelligence(schema=MyModel)",
    "for_regulated(jurisdiction='hipaa')",
    "for_medical()",
    "for_legal()",
    "for_financial()",
]:
    print(f"  EvalSuite.{name}")

---
## Next steps

**More notebooks:**
- [RAG Evaluation](rag_eval.ipynb) — Faithfulness, ContextPrecision, ContextRecall, experiment comparison
- [Agent Evaluation](agent_eval.ipynb) — ManualTracer, ToolCallAccuracy, flakiness detection

**Docs:**
- **CheckEvaluator:** plain-English checks with pinnable questions → [CheckEvaluator docs](https://evaldocs.multivon.ai/evaluators/llm-judge#checkevaluator)
- **Agent evaluators:** tool call accuracy, trajectory efficiency, plan quality → [Agent Evaluators](https://evaldocs.multivon.ai/evaluators/agent)
- **Experiment tracking:** compare runs with p-values → [Experiment Tracking](https://evaldocs.multivon.ai/guides/experiments)
- **CI/CD integration:** block deploys on regression → [CI/CD Guide](https://evaldocs.multivon.ai/guides/ci-cd)
- **Local LLM judge:** Ollama, LM Studio, vLLM → [JudgeConfig docs](https://evaldocs.multivon.ai/evaluators/llm-judge#local-and-self-hosted-models)
- **Compliance:** HIPAA PII detection, EU AI Act audit trails → [Compliance Guide](https://evaldocs.multivon.ai/guides/compliance)
- **GitHub:** [github.com/multivon-ai/multivon-eval](https://github.com/multivon-ai/multivon-eval)

**Found a bug or want a feature?** Open an issue: [github.com/multivon-ai/multivon-eval/issues](https://github.com/multivon-ai/multivon-eval/issues)